# 1. Introduction and scope

This notebook implements the PRD as a pure comparison-and-decision workflow for the one-month direction classifier.

Inputs expected:
- Six standardized metrics JSON files from 3_evaluate_model.
- User-defined thresholds and ranking policy.

Logic in the paired code cell:
- Initialize run context metadata and a scope flag that disallows any model training or tuning execution.

Outputs:
- A run context object used by downstream sections.

In [3]:
from datetime import datetime, timezone

run_context = {
    "notebook_name": "4_select_model.ipynb",
    "purpose": "Pure comparison-and-decision model selection workflow",
    "mode": "comparison_only",
    "allow_model_training": False,
    "allow_hyperparameter_search_execution": False,
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
}

print("Run context initialized")
print(f"Notebook mode: {run_context['mode']}")


Run context initialized
Notebook mode: comparison_only


## 1.1 Imports and user-defined variablesThis step defines required libraries and all configurable inputs for the notebook.Inputs expected:- A workspace that contains a project directory with both 3_evaluate_model and 4_select_model subfolders.Logic in the paired code cell:- Load libraries for paths, JSON, tables, and notebook display.- Discover the project directory robustly by scanning current and parent paths for the expected folder structure.- Define JSON input source (explicit file list or directory plus glob).- Define output directory 4_select_model.- Define primary metric policy (validation macro F1 first, test macro F1 guardrail).- Define thresholds: test_f1_buy >= 0.20, test_f1_sell >= 0.15, optional recall thresholds, and budget preset (small or medium or large).Outputs:- Centralized configuration object for the rest of the notebook.

In [6]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Resolve project paths robustly regardless of current working directory.
workspace_root = Path.cwd()
candidate_roots = [workspace_root] + list(workspace_root.parents)
project_dir = None
for root in candidate_roots:
    direct_match = root / "1-month-direction-classifier"
    if direct_match.exists() and (direct_match / "3_evaluate_model").exists() and (direct_match / "4_select_model").exists():
        project_dir = direct_match
        break

    for child in root.iterdir() if root.exists() else []:
        if not child.is_dir():
            continue
        if (child / "3_evaluate_model").exists() and (child / "4_select_model").exists():
            project_dir = child
            break
    if project_dir is not None:
        break

if project_dir is None:
    raise FileNotFoundError("Could not locate project directory containing both 3_evaluate_model and 4_select_model.")

input_dir = project_dir / "3_evaluate_model"
output_dir = project_dir / "4_select_model"
output_dir.mkdir(parents=True, exist_ok=True)

config = {
    "input": {
        "explicit_json_paths": [],
        "metrics_glob": "*_baseline_metrics.json",
        "expected_artifact_count": 6,
    },
    "paths": {
        "workspace_root": str(workspace_root),
        "project_dir": str(project_dir),
        "input_dir": str(input_dir),
        "output_dir": str(output_dir),
    },
    "selection_policy": {
        "primary_metric": "val_macro_f1",
        "guardrail_metric": "test_macro_f1",
        "tie_breakers": ["test_macro_f1", "test_balanced_accuracy"],
        "dominance_margin": 0.02,
        "stability_delta_threshold": 0.05,
    },
    "thresholds": {
        "hard": {
            "test_f1_buy_min": 0.20,
            "test_f1_sell_min": 0.15,
        },
        "optional_recall": {
            "enabled": True,
            "test_recall_buy_min": 0.18,
            "test_recall_sell_min": 0.13,
        },
    },
    "budget_preset": "medium",
    "budget_presets": {
        "small": {"n_iter": 20, "grid_size": "narrow"},
        "medium": {"n_iter": 50, "grid_size": "standard"},
        "large": {"n_iter": 100, "grid_size": "wide"},
    },
}

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)

display(Markdown("### Configuration"))
display(pd.json_normalize(config, sep="."))


### Configuration

,budget_preset,input.explicit_json_paths,input.metrics_glob,input.expected_artifact_count,paths.workspace_root,paths.project_dir,paths.input_dir,paths.output_dir,selection_policy.primary_metric,selection_policy.guardrail_metric,selection_policy.tie_breakers,selection_policy.dominance_margin,selection_policy.stability_delta_threshold,thresholds.hard.test_f1_buy_min,thresholds.hard.test_f1_sell_min,thresholds.optional_recall.enabled,thresholds.optional_recall.test_recall_buy_min,thresholds.optional_recall.test_recall_sell_min,budget_presets.small.n_iter,budget_presets.small.grid_size,budget_presets.medium.n_iter,budget_presets.medium.grid_size,budget_presets.large.n_iter,budget_presets.large.grid_size
0,medium,[],*_baseline_metrics.json,6,c:\Users\fraggable\DCC_Created\Coding\algorith...,c:\Users\fraggable\DCC_Created\Coding\algorith...,c:\Users\fraggable\DCC_Created\Coding\algorith...,c:\Users\fraggable\DCC_Created\Coding\algorith...,val_macro_f1,test_macro_f1,"[test_macro_f1, test_balanced_accuracy]",0.02,0.05,0.2,0.15,True,0.18,0.13,20,narrow,50,standard,100,wide


# 2. Artifact discovery, loading, and schema validationThis section finds baseline metric artifacts, loads them, and validates schema quality before comparison.Inputs expected:- Configuration object with path and pattern settings.Logic in the paired code cell:- Resolve JSON files and enforce the expected baseline set size (six files in the current phase).- Enforce path safety for explicit JSON paths by constraining inputs to the project directory.- Parse all artifacts into memory and attach source-file metadata.- Validate required top-level keys and nested metric keys for val and test, including per-class buy hold sell fields.- Build a validation report and stop if blocking schema issues are present.Outputs:- Validated artifact collection and schema validation report.

In [7]:
required_top_keys = {"model_name", "model_type", "hyperparameters", "data", "baseline", "metrics", "notes"}required_split_keys = {"macro_f1", "weighted_f1", "balanced_accuracy", "per_class", "confusion_matrix"}required_classes = {"buy", "hold", "sell"}required_class_metric_keys = {"precision", "recall", "f1"}project_root = Path(config["paths"]["project_dir"]).resolve()def _validate_explicit_metric_paths(explicit_paths, project_dir):    validated = []    for raw_path in explicit_paths:        p = Path(raw_path).expanduser().resolve()        if not p.exists() or not p.is_file():            raise FileNotFoundError(f"Explicit metrics path does not exist or is not a file: {p}")        if p.suffix.lower() != ".json":            raise ValueError(f"Explicit metrics path must be a JSON file: {p}")        if not p.is_relative_to(project_dir):            raise ValueError(                f"Explicit metrics path must be inside the project directory ({project_dir}): {p}"            )        validated.append(p)    return validateddef resolve_metric_files(cfg):    explicit = cfg["input"]["explicit_json_paths"]    if explicit:        return _validate_explicit_metric_paths(explicit, project_root)    return sorted(Path(cfg["paths"]["input_dir"]).glob(cfg["input"]["metrics_glob"]))metrics_files = resolve_metric_files(config)expected_count = config["input"]["expected_artifact_count"]if len(metrics_files) != expected_count:    raise ValueError(        f"Expected {expected_count} baseline metrics files, found {len(metrics_files)}. "        f"Files: {[p.name for p in metrics_files]}"    )loaded_artifacts = []validation_issues = []for path in metrics_files:    try:        with path.open("r", encoding="utf-8") as f:            payload = json.load(f)    except Exception as exc:        validation_issues.append({            "file": str(path),            "severity": "error",            "issue": f"Failed to parse JSON: {exc}",        })        continue    missing_top = sorted(required_top_keys - set(payload.keys()))    if missing_top:        validation_issues.append({            "file": str(path),            "severity": "error",            "issue": f"Missing top-level keys: {missing_top}",        })        continue    artifact_errors = []    for split in ["val", "test"]:        split_dict = payload.get("metrics", {}).get(split, {})        missing_split_keys = sorted(required_split_keys - set(split_dict.keys()))        if missing_split_keys:            artifact_errors.append(f"metrics.{split} missing keys: {missing_split_keys}")            continue        per_class = split_dict.get("per_class", {})        missing_classes = sorted(required_classes - set(per_class.keys()))        if missing_classes:            artifact_errors.append(f"metrics.{split}.per_class missing classes: {missing_classes}")            continue        for cls in sorted(required_classes):            cls_metrics = per_class.get(cls, {})            missing_cls_metrics = sorted(required_class_metric_keys - set(cls_metrics.keys()))            if missing_cls_metrics:                artifact_errors.append(                    f"metrics.{split}.per_class.{cls} missing keys: {missing_cls_metrics}"                )    if artifact_errors:        for err in artifact_errors:            validation_issues.append({                "file": str(path),                "severity": "error",                "issue": err,            })        continue    loaded_artifacts.append({        "source_file": str(path),        "source_name": path.name,        "payload": payload,    })schema_validation_report = pd.DataFrame(validation_issues)if not schema_validation_report.empty:    display(Markdown("### Schema validation issues"))    display(schema_validation_report)blocking_errors = schema_validation_report.query("severity == 'error'") if not schema_validation_report.empty else pd.DataFrame()if not blocking_errors.empty:    raise ValueError("Blocking schema validation errors detected. Resolve issues before proceeding.")validated_artifacts = loaded_artifactsprint(f"Validated artifacts: {len(validated_artifacts)}")print("Files:", [a["source_name"] for a in validated_artifacts])

Validated artifacts: 6
Files: ['extra_trees_baseline_metrics.json', 'hist_gradient_boosting_baseline_metrics.json', 'linear_svc_baseline_metrics.json', 'logistic_regression_baseline_metrics.json', 'mlp_classifier_baseline_metrics.json', 'random_forest_baseline_metrics.json']


# 3. Leaderboard normalization

This section converts each JSON artifact into a single normalized model row.

Inputs expected:
- Validated artifacts from Section 2.

Logic in the paired code cell:
- Flatten val and test metrics into a compact structure with one row per model.
- Extract macro F1, weighted F1, balanced accuracy, and per-class buy hold sell F1 and recall.
- Add metadata fields including model_name, model_type, class_weight_strategy notes, and source file path.

Outputs:
- Canonical leaderboard DataFrame with consistent column names and data types.

In [8]:
def _safe_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan

normalized_rows = []

for item in validated_artifacts:
    p = item["payload"]
    notes = p.get("notes", {})

    row = {
        "model_name": p.get("model_name"),
        "model_type": p.get("model_type"),
        "source_file": item["source_file"],
        "class_weight_strategy": notes.get("class_weight_strategy"),
        "hyperparameters": p.get("hyperparameters", {}),
    }

    for split in ["val", "test"]:
        split_metrics = p.get("metrics", {}).get(split, {})
        row[f"{split}_macro_f1"] = _safe_float(split_metrics.get("macro_f1"))
        row[f"{split}_weighted_f1"] = _safe_float(split_metrics.get("weighted_f1"))
        row[f"{split}_balanced_accuracy"] = _safe_float(split_metrics.get("balanced_accuracy"))

        per_class = split_metrics.get("per_class", {})
        for cls in ["buy", "hold", "sell"]:
            cls_metrics = per_class.get(cls, {})
            row[f"{split}_f1_{cls}"] = _safe_float(cls_metrics.get("f1"))
            row[f"{split}_recall_{cls}"] = _safe_float(cls_metrics.get("recall"))
            row[f"{split}_precision_{cls}"] = _safe_float(cls_metrics.get("precision"))

    normalized_rows.append(row)

leaderboard_df = pd.DataFrame(normalized_rows).sort_values("model_name").reset_index(drop=True)

numeric_cols = [c for c in leaderboard_df.columns if c.startswith("val_") or c.startswith("test_")]
leaderboard_df[numeric_cols] = leaderboard_df[numeric_cols].apply(pd.to_numeric, errors="coerce")

if leaderboard_df["model_name"].duplicated().any():
    dups = leaderboard_df.loc[leaderboard_df["model_name"].duplicated(), "model_name"].tolist()
    raise ValueError(f"Duplicate model_name values found: {dups}")

display(Markdown("### Canonical leaderboard DataFrame"))
display(leaderboard_df)


### Canonical leaderboard DataFrame

,model_name,model_type,source_file,class_weight_strategy,hyperparameters,val_macro_f1,val_weighted_f1,val_balanced_accuracy,val_f1_buy,val_recall_buy,val_precision_buy,val_f1_hold,val_recall_hold,val_precision_hold,val_f1_sell,val_recall_sell,val_precision_sell,test_macro_f1,test_weighted_f1,test_balanced_accuracy,test_f1_buy,test_recall_buy,test_precision_buy,test_f1_hold,test_recall_hold,test_precision_hold,test_f1_sell,test_recall_sell,test_precision_sell
0,extra_trees_baseline,sklearn.ensemble._forest.ExtraTreesClassifier,c:\Users\fraggable\DCC_Created\Coding\algorith...,balanced - mitigates hold-class dominance,"{'bootstrap': False, 'ccp_alpha': 0.0, 'class_...",0.4263,0.4320,0.4300,0.4246,0.3744,0.4903,0.4990,0.6066,0.4238,0.3554,0.3091,0.4180,0.3076,0.3600,0.3268,0.1710,0.1285,0.2556,0.5750,0.6840,0.4960,0.1767,0.1679,0.1864
1,hist_gradient_boosting_baseline,sklearn.ensemble._hist_gradient_boosting.gradi...,c:\Users\fraggable\DCC_Created\Coding\algorith...,none - HistGradientBoostingClassifier does not...,"{'categorical_features': 'from_dtype', 'class_...",0.4086,0.4101,0.4073,0.4541,0.4384,0.4709,0.3891,0.4076,0.3723,0.3827,0.3758,0.3899,0.3523,0.3875,0.3503,0.3590,0.3128,0.4211,0.4974,0.5242,0.4732,0.2007,0.2137,0.1892
2,linear_svc_baseline,sklearn.svm._classes.LinearSVC,c:\Users\fraggable\DCC_Created\Coding\algorith...,balanced - LinearSVC uses class_weight=balance...,"{'C': 1.0, 'class_weight': 'balanced', 'dual':...",0.3072,0.3074,0.3369,0.1310,0.0936,0.2184,0.4552,0.6019,0.3660,0.3355,0.3152,0.3586,0.3142,0.3657,0.3284,0.2353,0.1899,0.3091,0.5576,0.6654,0.4799,0.1498,0.1298,0.1771
3,logistic_regression_baseline,sklearn.linear_model._logistic.LogisticRegression,c:\Users\fraggable\DCC_Created\Coding\algorith...,balanced class weights via LogisticRegression(...,"{'C': 1.0, 'class_weight': 'balanced', 'dual':...",0.3112,0.3113,0.3279,0.1662,0.1330,0.2213,0.4330,0.5355,0.3633,0.3344,0.3152,0.3562,0.3311,0.3789,0.3378,0.2919,0.2626,0.3287,0.5455,0.6134,0.4911,0.1558,0.1374,0.1800
4,mlp_classifier_baseline,sklearn.neural_network._multilayer_perceptron....,c:\Users\fraggable\DCC_Created\Coding\algorith...,none - MLPClassifier does not support class_we...,"{'activation': 'relu', 'alpha': 0.001, 'batch_...",0.3398,0.3470,0.3478,0.4662,0.5271,0.4180,0.3269,0.3223,0.3317,0.2261,0.1939,0.2712,0.3617,0.3692,0.3786,0.4449,0.5754,0.3627,0.3640,0.3086,0.4439,0.2762,0.2519,0.3056
5,random_forest_baseline,sklearn.ensemble._forest.RandomForestClassifier,c:\Users\fraggable\DCC_Created\Coding\algorith...,balanced - mitigates hold-class dominance,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.4150,0.4159,0.4208,0.3792,0.3054,0.5000,0.4559,0.5024,0.4173,0.4098,0.4545,0.3731,0.2679,0.3173,0.2878,0.1061,0.0782,0.1647,0.5316,0.6097,0.4713,0.1661,0.1756,0.1575


# 4. Validation and test summary tables

This section builds the compact comparison tables required by the PRD.

Inputs expected:
- Canonical leaderboard DataFrame from Section 3.

Logic in the paired code cell:
- Build validation summary columns (val_macro_f1, val_weighted_f1, val_balanced_accuracy, val_f1_buy, val_f1_sell, optionally val_f1_hold).
- Build test summary columns (test_macro_f1, test_weighted_f1, test_balanced_accuracy, test_f1_buy, test_f1_sell, optionally test_f1_hold).
- Apply consistent display formatting for both tables.

Outputs:
- Validation summary table and test summary table for side-by-side analysis.

In [9]:
validation_columns = [
    "model_name",
    "val_macro_f1",
    "val_weighted_f1",
    "val_balanced_accuracy",
    "val_f1_buy",
    "val_f1_sell",
    "val_f1_hold",
]

test_columns = [
    "model_name",
    "test_macro_f1",
    "test_weighted_f1",
    "test_balanced_accuracy",
    "test_f1_buy",
    "test_f1_sell",
    "test_f1_hold",
]

validation_summary_df = leaderboard_df[validation_columns].copy()
test_summary_df = leaderboard_df[test_columns].copy()

for df in [validation_summary_df, test_summary_df]:
    metric_cols = [c for c in df.columns if c != "model_name"]
    df[metric_cols] = df[metric_cols].round(4)

display(Markdown("### Validation summary"))
display(validation_summary_df.sort_values("val_macro_f1", ascending=False).reset_index(drop=True))

display(Markdown("### Test summary"))
display(test_summary_df.sort_values("test_macro_f1", ascending=False).reset_index(drop=True))


### Validation summary

,model_name,val_macro_f1,val_weighted_f1,val_balanced_accuracy,val_f1_buy,val_f1_sell,val_f1_hold
0,extra_trees_baseline,0.4263,0.4320,0.4300,0.4246,0.3554,0.4990
1,random_forest_baseline,0.4150,0.4159,0.4208,0.3792,0.4098,0.4559
2,hist_gradient_boosting_baseline,0.4086,0.4101,0.4073,0.4541,0.3827,0.3891
3,mlp_classifier_baseline,0.3398,0.3470,0.3478,0.4662,0.2261,0.3269
4,logistic_regression_baseline,0.3112,0.3113,0.3279,0.1662,0.3344,0.4330
5,linear_svc_baseline,0.3072,0.3074,0.3369,0.1310,0.3355,0.4552


### Test summary

,model_name,test_macro_f1,test_weighted_f1,test_balanced_accuracy,test_f1_buy,test_f1_sell,test_f1_hold
0,mlp_classifier_baseline,0.3617,0.3692,0.3786,0.4449,0.2762,0.3640
1,hist_gradient_boosting_baseline,0.3523,0.3875,0.3503,0.3590,0.2007,0.4974
2,logistic_regression_baseline,0.3311,0.3789,0.3378,0.2919,0.1558,0.5455
3,linear_svc_baseline,0.3142,0.3657,0.3284,0.2353,0.1498,0.5576
4,extra_trees_baseline,0.3076,0.3600,0.3268,0.1710,0.1767,0.5750
5,random_forest_baseline,0.2679,0.3173,0.2878,0.1061,0.1661,0.5316


# 5. Ranking and robustness checks

This section applies ranking policy, tie-breakers, and validation-to-test stability checks.

Inputs expected:
- Validation and test summary tables.

Logic in the paired code cell:
- Compute rank_val_macro_f1 and rank_test_macro_f1 in descending order.
- Apply deterministic tie-break logic (test macro F1, then balanced accuracy).
- Compute deltas between validation and test for macro F1 and buy and sell F1.
- Add overfitting or instability flags based on configurable delta thresholds.

Outputs:
- Ranked leaderboard with stability diagnostics.

Open question:
- The PRD does not specify one fixed numeric instability threshold; this should remain configurable and logged in outputs.

In [10]:
ranked_leaderboard_df = leaderboard_df.copy()

ranked_leaderboard_df = ranked_leaderboard_df.sort_values(
    by=[
        config["selection_policy"]["primary_metric"],
        config["selection_policy"]["guardrail_metric"],
        "test_balanced_accuracy",
    ],
    ascending=[False, False, False],
).reset_index(drop=True)

ranked_leaderboard_df["rank_val_macro_f1"] = ranked_leaderboard_df["val_macro_f1"].rank(method="min", ascending=False).astype(int)
ranked_leaderboard_df["rank_test_macro_f1"] = ranked_leaderboard_df["test_macro_f1"].rank(method="min", ascending=False).astype(int)

ranked_leaderboard_df["delta_macro_f1_val_minus_test"] = ranked_leaderboard_df["val_macro_f1"] - ranked_leaderboard_df["test_macro_f1"]
ranked_leaderboard_df["delta_f1_buy_val_minus_test"] = ranked_leaderboard_df["val_f1_buy"] - ranked_leaderboard_df["test_f1_buy"]
ranked_leaderboard_df["delta_f1_sell_val_minus_test"] = ranked_leaderboard_df["val_f1_sell"] - ranked_leaderboard_df["test_f1_sell"]

stability_threshold = float(config["selection_policy"]["stability_delta_threshold"])
ranked_leaderboard_df["flag_instability"] = (
    (ranked_leaderboard_df["delta_macro_f1_val_minus_test"] > stability_threshold)
    | (ranked_leaderboard_df["delta_f1_buy_val_minus_test"] > stability_threshold)
    | (ranked_leaderboard_df["delta_f1_sell_val_minus_test"] > stability_threshold)
)

display(Markdown("### Ranked leaderboard with robustness diagnostics"))
display(
    ranked_leaderboard_df[[
        "model_name",
        "rank_val_macro_f1",
        "rank_test_macro_f1",
        "val_macro_f1",
        "test_macro_f1",
        "delta_macro_f1_val_minus_test",
        "delta_f1_buy_val_minus_test",
        "delta_f1_sell_val_minus_test",
        "flag_instability",
    ]].round(4)
)


### Ranked leaderboard with robustness diagnostics

,model_name,rank_val_macro_f1,rank_test_macro_f1,val_macro_f1,test_macro_f1,delta_macro_f1_val_minus_test,delta_f1_buy_val_minus_test,delta_f1_sell_val_minus_test,flag_instability
0,extra_trees_baseline,1,5,0.4263,0.3076,0.1187,0.2536,0.1787,True
1,random_forest_baseline,2,6,0.4150,0.2679,0.1471,0.2731,0.2437,True
2,hist_gradient_boosting_baseline,3,2,0.4086,0.3523,0.0563,0.0951,0.1820,True
3,mlp_classifier_baseline,4,1,0.3398,0.3617,-0.0219,0.0213,-0.0501,False
4,logistic_regression_baseline,5,3,0.3112,0.3311,-0.0199,-0.1257,0.1786,True
5,linear_svc_baseline,6,4,0.3072,0.3142,-0.0070,-0.1043,0.1857,True


# 6. Action-class viability filters

This section enforces buy and sell guardrails to avoid models that mainly predict hold.

Inputs expected:
- Ranked leaderboard with per-class metrics and threshold configuration.

Logic in the paired code cell:
- Apply hard minimums: test_f1_buy >= 0.20 and test_f1_sell >= 0.15.
- Optionally apply test_recall_buy >= 0.18 and test_recall_sell >= 0.13 when enabled.
- Add eligibility flags and reason codes explaining each pass or fail outcome.

Outputs:
- Eligibility-enriched leaderboard used for decision logic.

In [11]:
thresholds = config["thresholds"]
hard = thresholds["hard"]
optional = thresholds["optional_recall"]

eligibility_leaderboard_df = ranked_leaderboard_df.copy()

eligibility_leaderboard_df["pass_hard_f1_buy"] = eligibility_leaderboard_df["test_f1_buy"] >= hard["test_f1_buy_min"]
eligibility_leaderboard_df["pass_hard_f1_sell"] = eligibility_leaderboard_df["test_f1_sell"] >= hard["test_f1_sell_min"]
eligibility_leaderboard_df["pass_hard_thresholds"] = (
    eligibility_leaderboard_df["pass_hard_f1_buy"] & eligibility_leaderboard_df["pass_hard_f1_sell"]
)

if optional["enabled"]:
    eligibility_leaderboard_df["pass_optional_recall_buy"] = eligibility_leaderboard_df["test_recall_buy"] >= optional["test_recall_buy_min"]
    eligibility_leaderboard_df["pass_optional_recall_sell"] = eligibility_leaderboard_df["test_recall_sell"] >= optional["test_recall_sell_min"]
    eligibility_leaderboard_df["pass_optional_recall"] = (
        eligibility_leaderboard_df["pass_optional_recall_buy"]
        & eligibility_leaderboard_df["pass_optional_recall_sell"]
    )
else:
    eligibility_leaderboard_df["pass_optional_recall_buy"] = True
    eligibility_leaderboard_df["pass_optional_recall_sell"] = True
    eligibility_leaderboard_df["pass_optional_recall"] = True

eligibility_leaderboard_df["eligible_for_tuning"] = (
    eligibility_leaderboard_df["pass_hard_thresholds"]
    & eligibility_leaderboard_df["pass_optional_recall"]
    & (~eligibility_leaderboard_df["flag_instability"])
)

def _reason_code(row):
    reasons = []
    if not row["pass_hard_f1_buy"]:
        reasons.append("fail_test_f1_buy")
    if not row["pass_hard_f1_sell"]:
        reasons.append("fail_test_f1_sell")
    if optional["enabled"] and not row["pass_optional_recall_buy"]:
        reasons.append("fail_test_recall_buy")
    if optional["enabled"] and not row["pass_optional_recall_sell"]:
        reasons.append("fail_test_recall_sell")
    if row["flag_instability"]:
        reasons.append("flag_instability")
    return "pass" if not reasons else "|".join(reasons)

eligibility_leaderboard_df["eligibility_reason_code"] = eligibility_leaderboard_df.apply(_reason_code, axis=1)

display(Markdown("### Action-class viability and eligibility"))
display(
    eligibility_leaderboard_df[[
        "model_name",
        "test_f1_buy",
        "test_f1_sell",
        "test_recall_buy",
        "test_recall_sell",
        "flag_instability",
        "eligible_for_tuning",
        "eligibility_reason_code",
    ]].round(4)
)


### Action-class viability and eligibility

,model_name,test_f1_buy,test_f1_sell,test_recall_buy,test_recall_sell,flag_instability,eligible_for_tuning,eligibility_reason_code
0,extra_trees_baseline,0.1710,0.1767,0.1285,0.1679,True,False,fail_test_f1_buy|fail_test_recall_buy|flag_ins...
1,random_forest_baseline,0.1061,0.1661,0.0782,0.1756,True,False,fail_test_f1_buy|fail_test_recall_buy|flag_ins...
2,hist_gradient_boosting_baseline,0.3590,0.2007,0.3128,0.2137,True,False,flag_instability
3,mlp_classifier_baseline,0.4449,0.2762,0.5754,0.2519,False,True,pass
4,logistic_regression_baseline,0.2919,0.1558,0.2626,0.1374,True,False,flag_instability
5,linear_svc_baseline,0.2353,0.1498,0.1899,0.1298,True,False,fail_test_f1_sell|fail_test_recall_sell|flag_i...


# 7. Baseline versus already_tuned governanceThis section applies the PRD governance rule for model status classification in the current baseline phase.Inputs expected:- Eligibility-enriched leaderboard from the previous section.Logic in the paired code cell:- Apply the current-phase rule that all six available artifacts are baseline by default.- Set is_baseline to True and already_tuned to False for all rows.- Add governance_note text explaining that no documented systematic TimeSeriesSplit search artifact is present in this phase.Outputs:- Governance status fields available for reporting and export.

In [12]:
governance_leaderboard_df = eligibility_leaderboard_df.copy()

# Current phase rule from PRD: mark all six current artifacts as baseline by default.
governance_leaderboard_df["is_baseline"] = True
governance_leaderboard_df["already_tuned"] = False

# Keep explicit governance rationale for traceability.
governance_leaderboard_df["governance_note"] = (
    "baseline by default in current phase; no documented systematic TimeSeriesSplit search artifact"
)

display(Markdown("### Governance status"))
display(
    governance_leaderboard_df[[
        "model_name",
        "is_baseline",
        "already_tuned",
        "governance_note",
    ]]
)


### Governance status

,model_name,is_baseline,already_tuned,governance_note
0,extra_trees_baseline,True,False,baseline by default in current phase; no docum...
1,random_forest_baseline,True,False,baseline by default in current phase; no docum...
2,hist_gradient_boosting_baseline,True,False,baseline by default in current phase; no docum...
3,mlp_classifier_baseline,True,False,baseline by default in current phase; no docum...
4,logistic_regression_baseline,True,False,baseline by default in current phase; no docum...
5,linear_svc_baseline,True,False,baseline by default in current phase; no docum...


# 8. Decision logic for tuning candidates and provisional productionThis section converts ranking and filters into explicit decisions.Inputs expected:- Eligibility-enriched leaderboard with ranks and governance status.Logic in the paired code cell:- Select one to three tuning candidates using validation-first ranking constrained by hard action-class filters and stability checks.- Select one provisional production candidate using validation-first ranking with test guardrail review.- If no model passes the eligibility filters, record no tuning candidates and no provisional production candidate, with an explicit explanation.- Generate concise decision rationale fields for each selected model.Outputs:- tuning_candidates list and provisional_production_candidate value.- decision_policy_note text describing whether a clear leader was found or no eligible model was available.Open question:- The phrase clearly superior is ambiguous in the PRD. Keep a configurable dominance margin and log it explicitly.

In [13]:
decision_df = governance_leaderboard_df.copy()dominance_margin = float(config["selection_policy"]["dominance_margin"])eligible_df = decision_df[decision_df["eligible_for_tuning"]].copy()eligible_df = eligible_df.sort_values(    by=["val_macro_f1", "test_macro_f1", "test_balanced_accuracy"],    ascending=[False, False, False],).reset_index(drop=True)if eligible_df.empty:    # PRD behavior: do not promote a fallback provisional model when no model is eligible.    tuning_candidates = []    provisional_production_candidate = None    decision_policy_note = (        "No eligible models passed filters; no tuning candidates or provisional production candidate selected."    )else:    tuning_candidates = eligible_df.head(3)["model_name"].tolist()    top_row = eligible_df.iloc[0]    provisional_production_candidate = top_row["model_name"]    if len(eligible_df) > 1:        second_row = eligible_df.iloc[1]        val_gap = top_row["val_macro_f1"] - second_row["val_macro_f1"]        test_gap = top_row["test_macro_f1"] - second_row["test_macro_f1"]        clearly_superior = (val_gap >= dominance_margin) and (test_gap >= 0)    else:        clearly_superior = True    decision_policy_note = (        "Top eligible model treated as clearly superior under configured dominance margin."        if clearly_superior        else "No clearly superior model under configured dominance margin; keep multiple tuning candidates."    )decision_df["selected_for_tuning"] = decision_df["model_name"].isin(tuning_candidates)decision_df["is_provisional_production_candidate"] = (    decision_df["model_name"] == provisional_production_candidate)def _decision_reason(row):    reasons = []    if row["selected_for_tuning"]:        reasons.append("selected_for_tuning")    if row["is_provisional_production_candidate"]:        reasons.append("provisional_production_candidate")    if not reasons:        reasons.append("not_selected")    reasons.append(row["eligibility_reason_code"])    return "|".join(reasons)decision_df["decision_rationale"] = decision_df.apply(_decision_reason, axis=1)display(Markdown("### Decision outputs"))print("Tuning candidates:", tuning_candidates)print("Provisional production candidate:", provisional_production_candidate)print("Decision policy note:", decision_policy_note)display(    decision_df[[        "model_name",        "rank_val_macro_f1",        "rank_test_macro_f1",        "eligible_for_tuning",        "selected_for_tuning",        "is_provisional_production_candidate",        "decision_rationale",    ]])

### Decision outputs

Tuning candidates: ['mlp_classifier_baseline']
Provisional production candidate: mlp_classifier_baseline
Decision policy note: Top eligible model treated as clearly superior under configured dominance margin.


,model_name,rank_val_macro_f1,rank_test_macro_f1,eligible_for_tuning,selected_for_tuning,is_provisional_production_candidate,decision_rationale
0,extra_trees_baseline,1,5,False,False,False,not_selected|fail_test_f1_buy|fail_test_recall...
1,random_forest_baseline,2,6,False,False,False,not_selected|fail_test_f1_buy|fail_test_recall...
2,hist_gradient_boosting_baseline,3,2,False,False,False,not_selected|flag_instability
3,mlp_classifier_baseline,4,1,True,True,True,selected_for_tuning|provisional_production_can...
4,logistic_regression_baseline,5,3,False,False,False,not_selected|flag_instability
5,linear_svc_baseline,6,4,False,False,False,not_selected|fail_test_f1_sell|fail_test_recal...


# 9. Tuning recommendation mini-spec generationThis section prepares model-specific recommendations for a future tuning notebook without executing search.Inputs expected:- tuning_candidates list, model_type values, current hyperparameters, and budget preset.Logic in the paired code cell:- Define model-type-specific parameter spaces for RandomForest, ExtraTrees, HistGradientBoosting, LogisticRegression, LinearSVC, and MLPClassifier.- Apply budget preset logic (small, medium, large) to set search breadth and depth.- Produce per-candidate mini-spec entries with TimeSeriesSplit recommendation (currently implemented as 4 folds), search method, and validation macro F1 scoring target.Outputs:- Structured tuning recommendation block for downstream notebooks and JSON export.Open question:- Exact trial counts per budget preset are not fixed by the PRD and should be defined as explicit defaults in configuration.

In [14]:
model_space_templates = {
    "RandomForestClassifier": {
        "n_estimators": [200, 400, 800],
        "max_depth": [6, 10, 16, None],
        "min_samples_leaf": [1, 3, 5, 10],
        "max_features": ["sqrt", "log2", 0.5],
        "class_weight": [None, "balanced", "balanced_subsample"],
    },
    "ExtraTreesClassifier": {
        "n_estimators": [200, 400, 800],
        "max_depth": [6, 10, 16, None],
        "min_samples_leaf": [1, 3, 5, 10],
        "max_features": ["sqrt", "log2", 0.5],
        "class_weight": [None, "balanced"],
    },
    "HistGradientBoostingClassifier": {
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "max_depth": [None, 4, 6, 8],
        "max_leaf_nodes": [15, 31, 63],
        "max_iter": [200, 400, 800],
        "l2_regularization": [0.0, 0.01, 0.1, 1.0],
        "min_samples_leaf": [10, 20, 40],
    },
    "LogisticRegression": {
        "C": [0.01, 0.1, 1.0, 3.0, 10.0],
        "penalty": ["l2"],
        "class_weight": [None, "balanced"],
        "solver": ["lbfgs", "saga"],
    },
    "LinearSVC": {
        "C": [0.01, 0.1, 1.0, 3.0, 10.0],
        "loss": ["squared_hinge"],
        "class_weight": [None, "balanced"],
        "max_iter": [3000, 5000, 8000],
    },
    "MLPClassifier": {
        "hidden_layer_sizes": [(64,), (128,), (64, 32), (128, 64)],
        "alpha": [0.0001, 0.001, 0.01],
        "learning_rate_init": [0.0005, 0.001, 0.003],
        "batch_size": [32, 64, 128],
        "early_stopping": [True],
    },
}

def _resolve_template_key(model_type: str):
    for key in model_space_templates.keys():
        if key in model_type:
            return key
    return None

budget = config["budget_presets"][config["budget_preset"]]

tuning_recommendations = {}
for model_name in tuning_candidates:
    row = decision_df.loc[decision_df["model_name"] == model_name].iloc[0]
    template_key = _resolve_template_key(str(row["model_type"]))
    tuning_recommendations[model_name] = {
        "model_type": row["model_type"],
        "template_key": template_key,
        "current_hyperparameters": row["hyperparameters"],
        "recommended_search_space": model_space_templates.get(template_key, {}),
        "budget_preset": config["budget_preset"],
        "budget_details": budget,
        "search_method": "RandomizedSearchCV",
        "time_series_cv": {"strategy": "TimeSeriesSplit", "n_splits": 4},
        "scoring_metric": "f1_macro",
        "selection_rule": "Keep hyperparameters with best validation macro F1",
    }

recommendation_rows = []
for m, rec in tuning_recommendations.items():
    recommendation_rows.append({
        "model_name": m,
        "model_type": rec["model_type"],
        "budget_preset": rec["budget_preset"],
        "search_method": rec["search_method"],
        "n_splits": rec["time_series_cv"]["n_splits"],
        "n_search_dimensions": len(rec["recommended_search_space"]),
    })

tuning_recommendations_df = pd.DataFrame(recommendation_rows)
display(Markdown("### Tuning recommendation mini-spec"))
display(tuning_recommendations_df)


### Tuning recommendation mini-spec

,model_name,model_type,budget_preset,search_method,n_splits,n_search_dimensions
0,mlp_classifier_baseline,sklearn.neural_network._multilayer_perceptron....,medium,RandomizedSearchCV,4,5


# 10. Output assembly and persistence

This section assembles required output artifacts and writes them to 4_select_model.

Inputs expected:
- Final leaderboard, decisions, tuning recommendations, and selection rules.

Logic in the paired code cell:
- Build comparison object with models, ranks, tuning_candidates, provisional_production_candidate, selection_rules, and notes.
- Save model_comparison_baselines.json.
- Save model_leaderboard_baselines.csv.
- Validate successful writes and capture output paths.

Outputs:
- Two persisted artifacts under 4_select_model required by the PRD.

In [15]:
selection_rules = {
    "primary_metric": "test_macro_f1",
    "constraints": {
        "min_test_f1_buy": config["thresholds"]["hard"]["test_f1_buy_min"],
        "min_test_f1_sell": config["thresholds"]["hard"]["test_f1_sell_min"],
        "stability_delta_threshold": config["selection_policy"]["stability_delta_threshold"],
    },
    "secondary_criteria": [
        "model_simplicity",
        "training_time",
        "inference_cost",
    ],
}

export_columns = [
    "model_name", "model_type",
    "val_macro_f1", "test_macro_f1",
    "val_weighted_f1", "test_weighted_f1",
    "val_balanced_accuracy", "test_balanced_accuracy",
    "val_f1_buy", "test_f1_buy",
    "val_f1_sell", "test_f1_sell",
    "val_recall_buy", "test_recall_buy",
    "val_recall_sell", "test_recall_sell",
    "rank_val_macro_f1", "rank_test_macro_f1",
    "flag_instability", "eligible_for_tuning",
    "eligibility_reason_code",
    "is_baseline", "already_tuned",
    "selected_for_tuning", "is_provisional_production_candidate",
    "decision_rationale",
]

leaderboard_export_df = decision_df[export_columns].copy()

comparison = {
    "generated_at_utc": run_context["started_at_utc"],
    "notebook_name": run_context["notebook_name"],
    "models": leaderboard_export_df.replace({np.nan: None}).to_dict(orient="records"),
    "tuning_candidates": tuning_candidates,
    "provisional_production_candidate": provisional_production_candidate,
    "selection_rules": selection_rules,
    "decision_policy_note": decision_policy_note,
    "tuning_recommendations": tuning_recommendations,
    "notes": "All models currently baseline; tuning to follow in dedicated notebook.",
}

comparison_json_path = Path(config["paths"]["output_dir"]) / "model_comparison_baselines.json"
leaderboard_csv_path = Path(config["paths"]["output_dir"]) / "model_leaderboard_baselines.csv"

with comparison_json_path.open("w", encoding="utf-8") as f:
    json.dump(comparison, f, indent=2)

leaderboard_export_df.to_csv(leaderboard_csv_path, index=False)

if not comparison_json_path.exists() or not leaderboard_csv_path.exists():
    raise RuntimeError("Output artifact write validation failed.")

output_paths = {
    "comparison_json": str(comparison_json_path),
    "leaderboard_csv": str(leaderboard_csv_path),
}

display(Markdown("### Persisted output artifacts"))
display(pd.DataFrame([
    {"artifact": "comparison_json", "path": output_paths["comparison_json"]},
    {"artifact": "leaderboard_csv", "path": output_paths["leaderboard_csv"]},
]))


### Persisted output artifacts

,artifact,path
0,comparison_json,c:\Users\fraggable\DCC_Created\Coding\algorith...
1,leaderboard_csv,c:\Users\fraggable\DCC_Created\Coding\algorith...


# 11. Final summary and ambiguity log

This final section provides run conclusions and explicitly logs unresolved specification details.

Inputs expected:
- Validation outcomes, decision outputs, and saved artifact statuses.

Logic in the paired code cell:
- Display final summary of model ranking outcome, selected tuning candidates, provisional production candidate, and output file locations.
- Display open questions recorded during the run (instability threshold and dominance margin).

Outputs:
- Human-readable completion summary and a compact ambiguity checklist for follow-up clarification.

In [16]:
ambiguity_log = [
    {
        "topic": "instability_threshold",
        "status": "configured_default",
        "value": config["selection_policy"]["stability_delta_threshold"],
        "note": "PRD does not define one fixed numeric value.",
    },
    {
        "topic": "dominance_margin",
        "status": "configured_default",
        "value": config["selection_policy"]["dominance_margin"],
        "note": "PRD describes clearly superior but not exact margin.",
    },
    {
        "topic": "budget_trial_counts",
        "status": "configured_default",
        "value": config["budget_presets"][config["budget_preset"]],
        "note": "PRD defines presets conceptually but not exact counts.",
    },
]

final_summary = {
    "artifacts_discovered": len(metrics_files),
    "artifacts_validated": len(validated_artifacts),
    "tuning_candidates": tuning_candidates,
    "provisional_production_candidate": provisional_production_candidate,
    "outputs": output_paths,
}

display(Markdown("### Final run summary"))
display(pd.DataFrame([final_summary]))

display(Markdown("### Ambiguity log"))
display(pd.DataFrame(ambiguity_log))

print("Notebook implementation cells completed.")


### Final run summary

,artifacts_discovered,artifacts_validated,tuning_candidates,provisional_production_candidate,outputs
0,6,6,[mlp_classifier_baseline],mlp_classifier_baseline,{'comparison_json': 'c:\Users\fraggable\DCC_Cr...


### Ambiguity log

,topic,status,value,note
0,instability_threshold,configured_default,0.05,PRD does not define one fixed numeric value.
1,dominance_margin,configured_default,0.02,PRD describes clearly superior but not exact m...
2,budget_trial_counts,configured_default,"{'n_iter': 50, 'grid_size': 'standard'}",PRD defines presets conceptually but not exact...


Notebook implementation cells completed.
